In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import xgboost as xgb
from imblearn.over_sampling import SMOTE, RandomOverSampler
import pickle   


from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    accuracy_score,
    classification_report
)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [2]:
# load cấu hình .env
load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_SCHEMA = os.getenv("DB_SCHEMA_DWH")

In [3]:
def get_db_engine():
    "Create SQLAlchemy engine."
    return create_engine(
        f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True
    )

In [5]:
# Load data from db
def load_fact_input():
    "Load fact_input"
    engine = get_db_engine()

    query = f"""
        select symbol_key, period_date, period_type, metric_code, metric_value
        from {DB_SCHEMA}.fact_input
    """

    print("Querying database via SQLAlchemy...")
    df = pd.read_sql(text(query), engine)

    return df

In [6]:
df = load_fact_input()

Querying database via SQLAlchemy...


DatabaseError: Execution failed on sql '
        select symbol_key, period_date, period_type, metric_code, metric_value
        from dwh.fact_input
    ': (psycopg2.errors.UndefinedColumn) column "period_type" does not exist
LINE 2:         select symbol_key, period_date, period_type, metric_...
                                                ^
HINT:  Perhaps you meant to reference the column "fact_input.period_date".

[SQL: 
        select symbol_key, period_date, period_type, metric_code, metric_value
        from dwh.fact_input
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [7]:
df_copy = df.copy()
df = df_copy

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 560899 entries, 0 to 560898
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   symbol_key    560899 non-null  int64  
 1   period_date   560899 non-null  object 
 2   period_type   560899 non-null  str    
 3   metric_code   560899 non-null  str    
 4   metric_value  560899 non-null  float64
dtypes: float64(1), int64(1), object(1), str(2)
memory usage: 21.4+ MB


In [9]:
# pivot data to wide-format
def pivot_to_wide_format(df):
    df = df.copy()
    df["period_date"] = pd.to_datetime(df["period_date"])

    df_daily = df[df["period_type"] == "daily"].copy()

    df_wide = df_daily.pivot_table(
        index=["symbol_key", "period_date"],
        columns="metric_code",
        values="metric_value",
        aggfunc="first",
    ).reset_index()
    df_wide.columns.name = None

    return df_wide

In [10]:
df_wide = pivot_to_wide_format(df)
df_wide.info()

<class 'pandas.DataFrame'>
RangeIndex: 29521 entries, 0 to 29520
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype        
---  ------            --------------  -----        
 0   symbol_key        29521 non-null  int64        
 1   period_date       29521 non-null  datetime64[s]
 2   bb_percent_b_20   29521 non-null  float64      
 3   bb_width_norm     29521 non-null  float64      
 4   ema_12_norm       29521 non-null  float64      
 5   ema_26_norm       29521 non-null  float64      
 6   ma5_vs_ma20       29521 non-null  float64      
 7   ma_15_norm        29521 non-null  float64      
 8   ma_20_norm        29521 non-null  float64      
 9   ma_5_norm         29521 non-null  float64      
 10  ma_9_norm         29521 non-null  float64      
 11  macd_hist_norm    29521 non-null  float64      
 12  macd_line_norm    29521 non-null  float64      
 13  obv_zscore        29521 non-null  float64      
 14  return_1d         29521 non-null  float64      
 

In [11]:
df_wide = df_wide.dropna()

In [12]:
df_wide.info()

<class 'pandas.DataFrame'>
RangeIndex: 29521 entries, 0 to 29520
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype        
---  ------            --------------  -----        
 0   symbol_key        29521 non-null  int64        
 1   period_date       29521 non-null  datetime64[s]
 2   bb_percent_b_20   29521 non-null  float64      
 3   bb_width_norm     29521 non-null  float64      
 4   ema_12_norm       29521 non-null  float64      
 5   ema_26_norm       29521 non-null  float64      
 6   ma5_vs_ma20       29521 non-null  float64      
 7   ma_15_norm        29521 non-null  float64      
 8   ma_20_norm        29521 non-null  float64      
 9   ma_5_norm         29521 non-null  float64      
 10  ma_9_norm         29521 non-null  float64      
 11  macd_hist_norm    29521 non-null  float64      
 12  macd_line_norm    29521 non-null  float64      
 13  obv_zscore        29521 non-null  float64      
 14  return_1d         29521 non-null  float64      
 

In [13]:
#gán nhãn
df_wide["target"] = 0 #silent
df_wide.loc[df_wide["return_next_3d"] > 0.02, "target"] = 1 #buy
df_wide.loc[df_wide["return_next_3d"] < -0.02, "target"] = 2 #sell

#check data
df_wide["target"].value_counts()

target
0    17293
1     6709
2     5519
Name: count, dtype: int64

In [14]:
print(df_wide['return_next_3d'].describe(percentiles=[.1, .2, .25, .75, .8, .9]))
print(df_wide['target'].value_counts(normalize=True))

count    29521.000000
mean         0.002757
std          0.035247
min         -0.215953
10%         -0.032787
20%         -0.018519
25%         -0.014218
75%          0.017857
80%          0.023188
90%          0.040373
max          0.316151
Name: return_next_3d, dtype: float64
target
0    0.585786
1    0.227262
2    0.186952
Name: proportion, dtype: float64


In [15]:
# Features & target
X = df_wide.drop(columns=["symbol_key", "period_date", "target", "return_next_3d"])
y = df_wide["target"]

# Lưu đúng feature columns
feature_cols = X.columns.tolist()

# Split với stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE chỉ trên train
smote = SMOTE(random_state=42, sampling_strategy='auto')
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Model
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05
)

model.fit(X_train_sm, y_train_sm)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.58      0.63      3459
           1       0.33      0.35      0.34      1342
           2       0.27      0.38      0.32      1104

    accuracy                           0.49      5905
   macro avg       0.43      0.44      0.43      5905
weighted avg       0.53      0.49      0.51      5905



In [16]:
from sklearn.metrics import f1_score

# Dự đoán
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Tính toán
f1_train = f1_score(y_train, y_train_pred, average="macro")
f1_test = f1_score(y_test, y_test_pred, average="macro") # Sửa thứ tự: (y_true, y_pred)

# In kết quả theo định dạng dễ nhìn
print(f"{'Dataset':<10} | {'F1 Score':<10}")
print("-" * 25)
print(f"{'TRAIN':<10} | {f1_train:.4f}")
print(f"{'TEST':<10} | {f1_test:.4f}")
print("-" * 25)
print(f"Gap (Overfit): {f1_train - f1_test:.4f}")

Dataset    | F1 Score  
-------------------------
TRAIN      | 0.4783
TEST       | 0.4290
-------------------------
Gap (Overfit): 0.0492


In [17]:
# print features
importance = model.feature_importances_

feat_imp = pd.Series(importance, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print(feat_imp.head(20))

bb_width_norm       0.128977
return_1d           0.092612
ema_26_norm         0.078699
ema_12_norm         0.076060
return_3d           0.066352
rsi_14              0.054478
return_5d           0.051870
signal_line_norm    0.049671
macd_hist_norm      0.047089
ma_9_norm           0.045142
ma_20_norm          0.043861
ma_15_norm          0.043861
vol_ratio_20        0.043070
macd_line_norm      0.041407
ma5_vs_ma20         0.035534
ma_5_norm           0.035391
obv_zscore          0.034903
bb_percent_b_20     0.031024
dtype: float32


In [18]:
# model save
MODEL_VERSION = "v4"
with open("models/trading_model_latest.pkl", "wb") as f:
    pickle.dump(model, f)
with open(f"models/model_{MODEL_VERSION}.pkl", "wb") as f:
    pickle.dump(model, f)
with open(f"models/feature_cols.pkl", "wb") as f:
    pickle.dump(feature_cols, f)